# Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


from sklearn.linear_model import LogisticRegression

In [2]:
import sklearn, imblearn
print("sklearn:", sklearn.__version__, "imblearn:", imblearn.__version__)

sklearn: 1.7.2 imblearn: 0.14.0


# Keep this random state

In [3]:
RANDOM_STATE=42

# Splitting functions and custom IQRClipper

In [4]:
# Convert one-hot cohort flags into a single label
def label_subcohort(df):
    df = df.copy()
    df["cohort"] = np.select(
        [
            df["is_mi"] == 1,
            df["is_pcr"] == 1,
            df["is_cbs"] == 1
        ],
        ["MI", "PCI", "CABG"],
        default="Other"
    )
    return df

# 1) collapse race to a few stable bins
def collapse_race(r):
    if pd.isna(r): return "UNKNOWN/OTHER"
    r = str(r).upper()
    if r.startswith("WHITE"): return "WHITE"
    if r.startswith("BLACK"): return "BLACK"
    if r.startswith("ASIAN"): return "ASIAN"
    if "HISPANIC" in r:        return "HISPANIC"
    if "PORTUGUESE" in r:      return "PORTUGUESE"
    if "SOUTH AMERICAN" in r:  return "HISPANIC"  # sensible merge
    if "PACIFIC ISLANDER" in r:return "PACIFIC"
    if r in {"OTHER","UNKNOWN","UNABLE TO OBTAIN","PATIENT DECLINED TO ANSWER",
             "MULTIPLE RACE/ETHNICITY","AMERICAN INDIAN/ALASKA NATIVE",
             "WHITE - OTHER EUROPEAN","WHITE - EASTERN EUROPEAN","WHITE - RUSSIAN",
             "WHITE - BRAZILIAN","BLACK/CAPE VERDEAN","BLACK/CARIBBEAN ISLAND"}:
        return "UNKNOWN/OTHER"
    return "UNKNOWN/OTHER"

def build_subject_strata(df: pd.DataFrame,
                         min_bin_size: int = 5,
                         outcome_col: str = "y",
                         cohort_col: str = "cohort",
                         race_col: str = "race"):
    
    g = df.copy()
    
    g["race_collapsed"] = g[race_col].map(collapse_race)

    # Make a *stable* per-subject cohort label.
    # Option A: majority cohort across their stays
    subj_mode_cohort = (
        g.groupby(["subject_id", cohort_col]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id", cohort_col]]
         .rename(columns={cohort_col: "cohort_major"})
    )

    # Outcome as a subject-level flag (any positive stay)
    subj_y = (
        g.groupby("subject_id")[outcome_col]
         .max()
         .rename("y_any")
         .reset_index()
    )

    # Subject-level collapsed race: mode
    subj_race = (
        g.groupby(["subject_id","race_collapsed"]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id","race_collapsed"]]
    )

    subj = (
        subj_mode_cohort.merge(subj_race, on="subject_id")
                        .merge(subj_y, on="subject_id")
    )

    # Compose strata label; you can add age bands, etc., if you want more control
    subj["strata"] = (
        subj["cohort_major"].astype(str) + "|" +
        subj["race_collapsed"].astype(str) + "|" +
        subj["y_any"].astype(str)
    )

    # Merge rare strata into a single bin to avoid stratify errors
    counts = subj["strata"].value_counts()
    rare = counts[counts < min_bin_size].index
    subj.loc[subj["strata"].isin(rare), "strata"] = "OTHER_BIN"

    return subj[["subject_id","strata"]]

def remove_outlier_drop_gapdays_and_split(df_all: pd.DataFrame,
                                          val_and_test_size: float = 0.30,
                                          random_state: int = 42,
                                          outcome_col: str = "y",
                                          cohort_col: str = "cohort",
                                          race_col: str = "race"):
    """
    - Hard-remove obvious data-entry outliers.
    - Drop prev_gap_days (non-imputable by design).
    - Stratified split by *subject_id* using a patient-level strata label
      (cohort_major × collapsed race × y_any), with rare bins merged.
    - Returns row-level train/val/test DataFrames.
    """
    
    
    df = df_all.copy()
    if "cohort" not in df.columns:
        df = label_subcohort(df)

    # 0) hard outliers + drop prev_gap_days
    df = df[(df["glucose_mean"] < 999999) & (df["wbc_last_24h"] < 400)]
    if "prev_gap_days" in df.columns:
        df = df.drop(columns=["prev_gap_days"])

    # 1) build subject-level strata
    subj = build_subject_strata(df, min_bin_size=5,
                                outcome_col=outcome_col,
                                cohort_col=cohort_col,
                                race_col=race_col)

    # 2) split BY subject IDs into train vs temp (val+test) with stratify
    train_ids, temp_ids = train_test_split(
        subj["subject_id"].values,
        test_size=val_and_test_size,
        random_state=random_state,
        stratify=subj["strata"].values
    )

    # 3) from the temp pool, split val vs test (50/50) with stratify again
    temp = subj[subj["subject_id"].isin(temp_ids)]
    val_ids, test_ids = train_test_split(
        temp["subject_id"].values,
        test_size=0.5,
        random_state=random_state,
        stratify=temp["strata"].values
    )

    # 4) map IDs back to rows
    train_df = df[df["subject_id"].isin(train_ids)].copy()
    val_df   = df[df["subject_id"].isin(val_ids)].copy()
    test_df  = df[df["subject_id"].isin(test_ids)].copy()

    # Optional sanity checks
    for s in ["train","val","test"]:
        print(s, df[df.subject_id.isin(eval(f"{s}_ids"))][outcome_col].mean())

    return train_df, val_df, test_df


def examine_splits(df, df_type='train'):
    print('-'*25, f'EXAMINING {df_type} data', '-'*25)
    print(f'{df_type} data has {df.shape[0]} ICU stays across {df.subject_id.nunique()} patients.')
    print('-'*25, 'EXAMINE FIRST FEW INSTANCES', '-'*25)
    display(df.head())
    
    print('-'*25, 'CLASS DISTRIBUTION (%)', '-'*25)
    display(df.y.value_counts(normalize=True)*100)
    
    print('-'*25, f'RACE DISTRIBUTION (%) ({df.race.nunique()} races)', '-'*25)
    display(df.race.value_counts(normalize=True)*100)
    
    print('-'*25, 'COHORT DISTRIBUTION (%)', '-'*25)
    display(df.cohort.value_counts(normalize=True)*100)
    
    print('-'*25, 'AGE DISTRIBUTION', '-'*25)
    df.hist('age')


class IQRClipper(BaseEstimator, TransformerMixin):
    """
    Clips feature values based on the Interquartile Range (IQR).
    Values below Q1 - factor*IQR are set to that lower bound,
    and values above Q3 + factor*IQR are set to that upper bound.

    Parameters
    ----------
    factor : float, default=1.5
        Multiplier for the IQR range.
    """

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        Q1 = X_df.quantile(0.25)
        Q3 = X_df.quantile(0.75)
        IQR = Q3 - Q1
        self.lower_bounds_ = Q1 - self.factor * IQR
        self.upper_bounds_ = Q3 + self.factor * IQR
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        X_clipped = X_df.clip(lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1)
        # Return as same type as input
        return X_clipped if isinstance(X, pd.DataFrame) else X_clipped.to_numpy()

# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

In [5]:
# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

df = pd.read_csv("merged_chd_model_v2.csv")

train_df, val_df, test_df = remove_outlier_drop_gapdays_and_split(
    df,
    val_and_test_size = 0.3,
    random_state=RANDOM_STATE,
    outcome_col='y',
    cohort_col='cohort',
    race_col='race'
)

train 0.058043273753527753
val 0.06519823788546256
test 0.05415002219263205


Run this to see the distribution of train, val, and test splits

In [6]:
# examine_splits(train_df, df_type='train')
# examine_splits(val_df, df_type='val')
# examine_splits(test_df, df_type='test')

# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

In [7]:
# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

# Columns to drop from X
ID_COLS = ["subject_id", "hadm_id", "stay_id", "is_mi", "is_pcr", "is_cbs"]
DROP_COLS = [c for c in (ID_COLS + ["y"]) if c in train_df.columns]

# Build X_all / y_all and derive feature lists FROM X_all
X_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True).drop(columns=DROP_COLS)
y_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)["y"]


# define numeric / categorical feature sets
numeric_cols = X_all.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_all.columns if c not in numeric_cols]

# 1. DEFINE THE PIPELINE (PREPROCESSING & OVERSAMPLING & MODELLING)

To swap models, change the final pipeline step from LogisticRegression to your own estimator (e.g. MLP via scikeras). Keep preprocessing identical.

In [8]:
# 1. DEFINE THE PIPELINE (PREPROCESSING & OVERSAMPLING & MODELLING)

# ---- numeric preprocessing pipeline
num_pipe = Pipeline([
    ("iqr_clip", IQRClipper(factor=1.5)),                 # outlier handling
    ("imputer", SimpleImputer(strategy="median")),        # median imputation
    ("scaler", StandardScaler())                          # standardize
])

# ---- categorical preprocessing pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), # mode imputation
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

# ---- combine
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ],
    remainder="drop"
)

# # ---- pipeline with optional oversampling
# want_oversample = True
# steps = [("pre", preprocessor)]
# if want_oversample:
#     steps.append(
#         ("ros", RandomOverSampler(random_state=RANDOM_STATE))
#     )
# steps.append(("model", LogisticRegression(max_iter=500, random_state=RANDOM_STATE)))

# pipe = ImbPipeline(steps=steps)

In [9]:
X_trainval = X_all.copy()
y_trainval = y_all.copy()

X_test = test_df.drop(columns=DROP_COLS)
y_test = test_df["y"]

missing_cols = set(X_trainval.columns) - set(X_test.columns)
for c in missing_cols:
    X_test[c] = np.nan
X_test = X_test[X_trainval.columns]

In [10]:
# display evaluation
def evaluate(model, X_te, y_te, name="Model", thr=0.5):
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= thr).astype(int)
    print(f"\n=== {name} ===")
    print("ROC-AUC:", roc_auc_score(y_te, y_prob))
    print("PR-AUC :", average_precision_score(y_te, y_prob))
    print("Confusion Matrix:\n", confusion_matrix(y_te, y_pred))
    print(classification_report(y_te, y_pred, digits=3))

In [17]:

lasso_plain = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(
        penalty="l1", solver="liblinear", max_iter=2000, C=1.0,
        class_weight=None, random_state=RANDOM_STATE
    ))
])
lasso_plain.fit(X_trainval, y_trainval)

y_val_prob = lasso_plain.predict_proba(X_trainval)[:, 1]

from sklearn.metrics import f1_score

best_t, best_f1 = 0, 0
for t in np.linspace(0, 1, 100):
    y_pred = (y_val_prob >= t).astype(int)
    f1 = f1_score(y_trainval, y_pred)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print("Best threshold:", round(best_t, 3), "F1:", round(best_f1, 3))

y_test_prob = lasso_plain.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_t).astype(int)

evaluate(lasso_plain, X_test, y_test, "LASSO (no resampling, tuned threshold)")


Best threshold: 0.101 F1: 0.257

=== LASSO (no resampling, tuned threshold) ===
ROC-AUC: 0.7473440468955542
PR-AUC : 0.16134768149325118
Confusion Matrix:
 [[2130    1]
 [ 122    0]]
              precision    recall  f1-score   support

           0      0.946     1.000     0.972      2131
           1      0.000     0.000     0.000       122

    accuracy                          0.945      2253
   macro avg      0.473     0.500     0.486      2253
weighted avg      0.895     0.945     0.919      2253



In [18]:
# 2 LASSO (with resampling)
lasso_ros = ImbPipeline([
    ("pre", preprocessor),
    ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
    ("clf", LogisticRegression(
        penalty="l1", solver="saga", max_iter=2000, C=1.0,
        class_weight=None, random_state=RANDOM_STATE
    ))
])
lasso_ros.fit(X_trainval, y_trainval)

y_val_prob = lasso_ros.predict_proba(X_trainval)[:, 1]

from sklearn.metrics import f1_score

best_t, best_f1 = 0, 0
for t in np.linspace(0, 1, 100):
    y_pred = (y_val_prob >= t).astype(int)
    f1 = f1_score(y_trainval, y_pred)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print("Best threshold:", round(best_t, 3), "F1:", round(best_f1, 3))

y_test_prob = lasso_ros.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_t).astype(int)

evaluate(lasso_ros, X_test, y_test, "LASSO (ROS)")

Best threshold: 0.646 F1: 0.248

=== LASSO (ROS) ===
ROC-AUC: 0.7337815694932726
PR-AUC : 0.14178628589663855
Confusion Matrix:
 [[1455  676]
 [  39   83]]
              precision    recall  f1-score   support

           0      0.974     0.683     0.803      2131
           1      0.109     0.680     0.188       122

    accuracy                          0.683      2253
   macro avg      0.542     0.682     0.496      2253
weighted avg      0.927     0.683     0.769      2253



In [ ]:
# 3 LR (inverse weight)
invw_lr = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(
        penalty="l2", solver="lbfgs", max_iter=2000,
        class_weight="balanced", random_state=RANDOM_STATE
    ))
])
invw_lr.fit(X_trainval, y_trainval)

y_val_prob = invw_lr.predict_proba(X_trainval)[:, 1]

best_t, best_f1 = 0, 0
for t in np.linspace(0, 1, 100):
    y_pred = (y_val_prob >= t).astype(int)
    f1 = f1_score(y_trainval, y_pred)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print("Best threshold:", round(best_t, 3), "F1:", round(best_f1, 3))

y_test_prob = invw_lr.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= best_t).astype(int)

evaluate(invw_lr, X_test, y_test, "Inverse-weight LR (balanced)")

Best threshold: 0.6767676767676768 F1: 0.25030674846625767

=== Inverse-weight LR (balanced) ===
ROC-AUC: 0.7376087575293674
PR-AUC : 0.14283817654388645
Confusion Matrix:
 [[1465  666]
 [  38   84]]
              precision    recall  f1-score   support

           0      0.975     0.687     0.806      2131
           1      0.112     0.689     0.193       122

    accuracy                          0.688      2253
   macro avg      0.543     0.688     0.499      2253
weighted avg      0.928     0.688     0.773      2253

